# Notebook C — Adapter Validation and Submission Packaging — v1.1

Purpose:
- consume the adapter produced by Notebook B
- validate adapter files exist and rank is compatible
- create a minimal `submission.zip` containing only adapter files
- optionally run a small vLLM metric-like inference smoke test

This notebook is for validation and packaging. It does **not** train.

Expected adapter inputs:
- `/kaggle/working/adapter_smoke_v1_1`, if run in the same session as Notebook B
- or a Kaggle input/notebook output containing `adapter_smoke_v1_1/adapter_config.json` and `adapter_smoke_v1_1/adapter_model.safetensors`
- or a Kaggle input containing `adapter_smoke_v1_1.zip`

If adapter discovery fails after adding Notebook B as input, restart the Kaggle session and rerun from the top.


In [ ]:
import glob
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import pandas as pd

RUN_VLLM_VALIDATION = False  # Start with False for packaging-only validation; set True after packaging passes.
VALIDATION_N = 3

WORKING_DIR = Path('/kaggle/working')
ADAPTER_EXTRACT_DIR = WORKING_DIR / 'adapter_for_validation_v1_1'
SUBMISSION_ZIP = WORKING_DIR / 'submission.zip'
DEBUG_CSV = WORKING_DIR / 'notebook_c_debug_predictions.csv'

# Optional manual override. Leave as None normally.
# Example: MANUAL_ADAPTER_DIR = Path('/kaggle/input/notebooks/<user>/<notebook-slug>/adapter_smoke_v1_1')
MANUAL_ADAPTER_DIR = None

print('RUN_VLLM_VALIDATION:', RUN_VLLM_VALIDATION)
print('WORKING_DIR:', WORKING_DIR)
print('/kaggle/input exists:', Path('/kaggle/input').exists())


## 1) Inspect attached inputs

This diagnostic cell makes it obvious whether Kaggle has mounted Notebook B output. You should see paths containing `adapter_smoke_v1_1`.


In [ ]:
print('Top-level /kaggle/input entries:')
if Path('/kaggle/input').exists():
    for p in sorted(Path('/kaggle/input').iterdir()):
        print(' -', p)
else:
    print(' - /kaggle/input does not exist')

print('\nAdapter-related files visible under /kaggle/input and /kaggle/working:')
roots = [Path('/kaggle/input'), WORKING_DIR]
visible = []
for root in roots:
    if not root.exists():
        continue
    for pattern in ['**/adapter_config.json', '**/adapter_model.safetensors', '**/adapter_model.bin', '**/adapter_smoke_v1_1.zip', '**/submission.zip']:
        visible.extend(root.glob(pattern))

for p in sorted(set(visible))[:100]:
    print(' -', p)
print('Count:', len(set(visible)))


## 2) Discover adapter files

The final Kaggle submission must contain `adapter_config.json` and adapter weights. This cell searches current working files and attached Kaggle inputs, including nested Notebook output folders.


In [ ]:
def _is_adapter_dir(d: Path) -> bool:
    return (d / 'adapter_config.json').exists() and (
        (d / 'adapter_model.safetensors').exists() or (d / 'adapter_model.bin').exists()
    )

def discover_adapter_dir():
    candidate_dirs = []

    if MANUAL_ADAPTER_DIR is not None:
        candidate_dirs.append(Path(MANUAL_ADAPTER_DIR))

    # Same-session Notebook B output.
    candidate_dirs.append(WORKING_DIR / 'adapter_smoke_v1_1')

    # Common Kaggle Notebook-output layout.
    for root in [Path('/kaggle/input/notebooks'), Path('/kaggle/input'), WORKING_DIR]:
        if not root.exists():
            continue
        # Direct config search.
        for cfg in root.glob('**/adapter_config.json'):
            candidate_dirs.append(cfg.parent)
        # Folder-name search.
        for d in root.glob('**/adapter_smoke_v1_1'):
            if d.is_dir():
                candidate_dirs.append(d)

    # If only a zip is attached, extract it.
    zip_candidates = []
    for root in [Path('/kaggle/input/notebooks'), Path('/kaggle/input'), WORKING_DIR]:
        if root.exists():
            zip_candidates.extend(root.glob('**/adapter_smoke_v1_1.zip'))
            zip_candidates.extend(root.glob('**/submission.zip'))

    print('Zip candidates:')
    for z in sorted(set(zip_candidates))[:30]:
        print(' -', z)

    for zpath in sorted(set(zip_candidates)):
        try:
            if ADAPTER_EXTRACT_DIR.exists():
                shutil.rmtree(ADAPTER_EXTRACT_DIR)
            ADAPTER_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall(ADAPTER_EXTRACT_DIR)
            for cfg in ADAPTER_EXTRACT_DIR.glob('**/adapter_config.json'):
                candidate_dirs.insert(0, cfg.parent)
            print('Extracted adapter zip candidate:', zpath)
            break
        except Exception as e:
            print('Could not extract zip candidate:', zpath, repr(e))

    # Deduplicate while preserving order.
    seen = set()
    unique_dirs = []
    for d in candidate_dirs:
        d = Path(d)
        key = str(d)
        if key not in seen:
            seen.add(key)
            unique_dirs.append(d)

    print('\nAdapter directory candidates:')
    for d in unique_dirs[:80]:
        print(' -', d, 'exists=', d.exists(), 'is_adapter=', _is_adapter_dir(d) if d.exists() else False)

    for d in unique_dirs:
        if _is_adapter_dir(d):
            return d
    return None

ADAPTER_DIR = discover_adapter_dir()
if ADAPTER_DIR is None:
    raise FileNotFoundError(
        'Could not find adapter_config.json plus adapter weights. '
        'Notebook B output appears in the Kaggle sidebar only after it is mounted into /kaggle/input. '
        'Try: Save Version for Notebook B, add that output as input to Notebook C, restart this session, then rerun.'
    )

print('\nSelected ADAPTER_DIR:', ADAPTER_DIR)
print('Files:')
for p in sorted(ADAPTER_DIR.iterdir()):
    if p.is_file():
        print(' -', p.name, f'{p.stat().st_size/1024/1024:.2f} MB')


## 3) Validate adapter config and rank

The competition allows a LoRA adapter with maximum rank 32.


In [ ]:
cfg_path = ADAPTER_DIR / 'adapter_config.json'
with open(cfg_path, 'r', encoding='utf-8') as f:
    adapter_cfg = json.load(f)

print(json.dumps(adapter_cfg, indent=2)[:2000])

rank = adapter_cfg.get('r', adapter_cfg.get('rank', None))
if rank is not None:
    print('Detected LoRA rank:', rank)
    assert int(rank) <= 32, f'LoRA rank {rank} exceeds competition max rank 32'
else:
    print('Warning: could not find rank field `r` in adapter config; inspect manually.')

assert _is_adapter_dir(ADAPTER_DIR)
print('Adapter structure validation passed.')


## 4) Create minimal submission.zip

This creates a root-level zip with only adapter files. This is the final-style package shape; do not include notebooks, README, tokenizer debug files, or extra artifacts.


In [ ]:
if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

required_or_allowed = ['adapter_config.json', 'adapter_model.safetensors', 'adapter_model.bin']
with zipfile.ZipFile(SUBMISSION_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in required_or_allowed:
        p = ADAPTER_DIR / name
        if p.exists():
            zf.write(p, arcname=name)

with zipfile.ZipFile(SUBMISSION_ZIP, 'r') as zf:
    names = zf.namelist()

print('Created:', SUBMISSION_ZIP, f'{SUBMISSION_ZIP.stat().st_size/1024/1024:.2f} MB')
print('Zip contents:', names)
assert 'adapter_config.json' in names
assert ('adapter_model.safetensors' in names) or ('adapter_model.bin' in names)
assert all('/' not in n.strip('/') for n in names), 'Adapter files should be at zip root, not nested.'
print('Minimal submission.zip packaging validation passed.')


## 5) Utility functions matching the metric extraction/verification

These mirror the official metric logic: prefer `\boxed{}` extraction; binary strings compare exactly; numeric answers use tolerance.


In [ ]:
def extract_final_answer(text: str | None) -> str:
    if text is None:
        return 'NOT_FOUND'
    matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()
    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()
    matches = re.findall(r'-?\d+(?:\.\d+)?', text)
    if matches:
        return matches[-1]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'

def verify(stored_answer: str, predicted: str) -> bool:
    stored_answer = str(stored_answer).strip()
    predicted = str(predicted).strip()
    if re.fullmatch(r'[01]+', stored_answer):
        return predicted.lower() == stored_answer.lower()
    try:
        return math.isclose(float(stored_answer), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()


## 6) Build small validation set

Uses competition `train.csv` if attached; otherwise falls back to the v1.1 trace JSONL.


In [ ]:
def discover_train_csv():
    candidates = [
        Path('/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv'),
        Path('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv'),
        Path('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv'),
    ]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').glob('**/train.csv'))
    for p in candidates:
        if p.exists():
            return p
    return None

def discover_trace_jsonl():
    candidates = [WORKING_DIR / 'train_traces_v1_1.jsonl']
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').glob('**/train_traces_v1_1.jsonl'))
    for p in candidates:
        if p.exists():
            return p
    return None

train_csv = discover_train_csv()
trace_jsonl = discover_trace_jsonl()

if train_csv is not None:
    full_df = pd.read_csv(train_csv)
    validation_df = full_df.sample(min(VALIDATION_N, len(full_df)), random_state=42).reset_index(drop=True)
    print('Using train CSV validation source:', train_csv)
elif trace_jsonl is not None:
    records = []
    with open(trace_jsonl, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                user_content = rec['messages'][0]['content']
                prompt = user_content.split('\nPlease put your final answer inside')[0]
                records.append({'id': rec['id'], 'prompt': prompt, 'answer': rec['answer']})
    validation_df = pd.DataFrame(records).sample(min(VALIDATION_N, len(records)), random_state=42).reset_index(drop=True)
    print('Using trace JSONL validation source:', trace_jsonl)
else:
    raise FileNotFoundError('Could not find train.csv or train_traces_v1_1.jsonl for validation prompts.')

print(validation_df[['id', 'answer']].head())


## 7) Optional vLLM validation

This uses metric-like vLLM loading with LoRA enabled. It is intentionally tiny. If it passes, the adapter-serving path is much more credible.

Run this only after packaging-only validation passes. It follows the official metric-style bootstrap and uninstalls torch in this notebook session.


In [ ]:
if RUN_VLLM_VALIDATION:
    commands = [
        'uv pip uninstall -y torch torchvision torchaudio',
        'tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp',
        'chmod +x /tmp/triton/backends/nvidia/bin/ptxas',
        'chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell',
    ]
    for cmd in commands:
        print('Running:', cmd)
        subprocess.run(cmd, shell=True, check=True)
    sys.path.insert(0, '/tmp')
else:
    print('RUN_VLLM_VALIDATION=False; skipping metric bootstrap.')


In [ ]:
if RUN_VLLM_VALIDATION:
    import kagglehub
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
    print('MODEL_PATH:', MODEL_PATH)
    print('ADAPTER_DIR:', ADAPTER_DIR)

    os.environ['TRANSFORMERS_NO_TF'] = '1'
    os.environ['TRANSFORMERS_NO_FLAX'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    os.environ['TRITON_PTXAS_PATH'] = '/tmp/triton/backends/nvidia/bin/ptxas'

    llm = LLM(
        model=str(MODEL_PATH),
        tensor_parallel_size=1,
        max_num_seqs=4,
        gpu_memory_utilization=0.85,
        dtype='auto',
        max_model_len=8192,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=32,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
    )

    sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=512)
    tokenizer = llm.get_tokenizer()

    prompts = []
    for row in validation_df.itertuples(index=False):
        user_content = str(row.prompt) + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
        try:
            prompt = tokenizer.apply_chat_template(
                [{'role': 'user', 'content': user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )
        except Exception:
            prompt = user_content
        prompts.append(prompt)

    print('Generating validation outputs...')
    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=LoRARequest('adapter', 1, str(ADAPTER_DIR)),
    )

    debug_rows = []
    for row, output in zip(validation_df.itertuples(index=False), outputs):
        raw = output.outputs[0].text
        pred = extract_final_answer(raw)
        ok = verify(str(row.answer), pred)
        debug_rows.append({
            'id': row.id,
            'answer': row.answer,
            'prediction': pred,
            'match': ok,
            'has_boxed': '\\boxed{' in raw,
            'raw_output': raw,
        })

    debug_df = pd.DataFrame(debug_rows)
    debug_df.to_csv(DEBUG_CSV, index=False)
    print(debug_df[['id', 'answer', 'prediction', 'match', 'has_boxed']])
    print('Tiny validation accuracy:', debug_df['match'].mean())
    print('Saved debug CSV:', DEBUG_CSV)

    assert len(debug_df) > 0
    assert debug_df['raw_output'].str.len().min() > 0
    print('vLLM adapter load + generation smoke validation completed.')
else:
    print('Skipped vLLM validation. Packaging-only validation is complete.')


## Final checkpoint

If packaging passed, `submission.zip` exists. If vLLM validation passed, the adapter loads and generates under metric-like serving. Do not submit a 6-example smoke adapter for score unless deliberately testing submission validity.


In [ ]:
print('Final artifacts:')
for p in [SUBMISSION_ZIP, DEBUG_CSV]:
    print(' -', p, 'exists=', p.exists(), f'{p.stat().st_size/1024/1024:.2f} MB' if p.exists() else '')

print('Notebook C complete.')
